In [6]:

# 1. Imports and setup
import json
import re
import emoji
import asyncio
from unidecode import unidecode
from googletrans import Translator
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.data.path.append(r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
nltk.download('stopwords', download_dir=r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
nltk.download('popular', download_dir=r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')

# 2. Slang dictionary and utility functions
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "tdk": "tidak",
    "aja": "saja",
    # Add more slang words as needed
}

def replace_slang(text, slang_dict):
    words = text.split()
    return ' '.join([slang_dict.get(w, w) for w in words])

def remove_extra_chars(text):
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    return text

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "), language='id')

def remove_usernames(text):
    return re.sub(r'@\w+', '{USER}', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s{}]', '', text)

def replace_links(text):
    return re.sub(r'http[s]?://\S+|www\.\S+', '{LINK}', text)

def normalize_text(text):
    # Convert to ASCII, remove unsupported formatting
    return unidecode(str(text))

import re

def fix_obfuscated_words(text):
    # Replace numbers with letters only if surrounded by letters (not part of a digit sequence)
    def replacer(match):
        word = match.group()
        # Only replace if the word contains both letters and numbers, and not ending in a digit sequence
        # e.g. "b4ru" -> "baru", but "ulti300" stays "ulti300"
        # Replace only single digits surrounded by letters
        word = re.sub(r'(?<=\D)0(?=\D)', 'o', word)
        word = re.sub(r'(?<=\D)1(?=\D)', 'i', word)
        word = re.sub(r'(?<=\D)3(?=\D)', 'e', word)
        word = re.sub(r'(?<=\D)4(?=\D)', 'a', word)
        word = re.sub(r'(?<=\D)5(?=\D)', 's', word)
        # word = re.sub(r'(?<=\D)7(?=\D)', 't', word)
        return word

    # Apply only to words containing both letters and numbers
    return re.sub(r'\b\w*[a-zA-Z]+\w*\b', replacer, text)

def lowercase(text):
    text = fix_obfuscated_words(text)
    return text.lower()

[nltk_data] Downloading package stopwords to C:/Users/Marvel Wilbert
[nltk_data]     O/AppData/Roaming/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to C:/Users/Marvel
[nltk_data]    |     Wilbert O/AppData/Roaming/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to C:/Users/Marvel
[nltk_data]    |     Wilbert O/AppData/Roaming/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to C:/Users/Marvel
[nltk_data]    |     Wilbert O/AppData/Roaming/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to C:/Users/Marvel
[nltk_data]    |     Wilbert O/AppData/Roaming/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading

In [2]:
# 3. Async translation and preprocessing function
async def back_translate(text):
    async with Translator() as translator:
        # Detect source language automatically
        en_result = await translator.translate(text, dest='en')
        en_text = en_result.text
        id_result = await translator.translate(en_text, dest='id')
        id_text = id_result.text
        return id_text

async def preprocess_text(text, stop_words, slang_dict):
    # 1. Back translation
    text = await back_translate(text)
    # Lowercase before further processing
    text = text.lower()
    # 2. Remove stopwords
    tokens = word_tokenize(text, language='indonesian')
    text = ' '.join([w for w in tokens if w.lower() not in stop_words])
    # 3. Replace slang words
    text = replace_slang(text, slang_dict)
    # 4. Remove extra characters
    text = remove_extra_chars(text)
    # 5. Convert emojis to phrases
    text = convert_emojis(text)
    # 6. Remove usernames
    text = remove_usernames(text)
    # 7. Remove numbers
    text = remove_numbers(text)
    # 8. Remove punctuation
    text = remove_punctuation(text)
    # 9. Unidecode (normalize unicode to ASCII)
    import unidecode
    text = unidecode.unidecode(text)
    return text.strip()


In [7]:
# 1. Convert emojis to phrases
sample_text = "🔥🔥INFO BARU BOSS Q,NI LINK YG @SEDANG HITS & VIRAL!!! 🔥🔥\n🔥🔥 EVENT SLOT SCATTER DAN PERKALIAN TERBESAR 🔥🔥\nPROMO HANYA BULAN INI SAJA !!! COBA DEPO 50K PASTIKAN MAXWIN🔥🔥\nIDNSCORE 𝗠𝗘𝗠𝗕𝗘𝗥𝗜𝗞𝗔𝗡 𝗕𝗢𝗡𝗨𝗦 𝟭𝟬𝟬% 𝗦𝗟𝗢𝗧  DAFTAR :\nhttps://magic.ly/LGOGOAL_PROFIT_VIP"
text1 = convert_emojis(sample_text)
print(text1)

# 2. Replace links with {LINK}
text2 = replace_links(text1)
print(text2)

# 3. Back translation
async def demo_back_translate(text):
    result = await back_translate(text)
    print(result)

await demo_back_translate(text2)

# 4. Lowercase
text3 = lowercase(text2)
print(text3)

# 5. Remove stopwords
stop_words = set(stopwords.words('indonesian'))
tokens = text3.split()
text4 = ' '.join([w for w in tokens if w.lower() not in stop_words])
print(text4)

# 6. Replace slang words
text5 = replace_slang(text4, slang_dict)
print(text5)

# 7. Remove extra characters
text6 = remove_extra_chars(text5)
print(text6)

# 8. Remove usernames
text7 = remove_usernames(text6)
print(text7)

# 9. Remove numbers
text8 = remove_numbers(text7)
print(text8)

# 10. Remove punctuation
text9 = remove_punctuation(text8)
print(text9)

# 11. Unidecode (normalize unicode to ASCII)
text10 = normalize_text(text9)
print(text10)


 api  api INFO BARU BOSS Q,NI LINK YG @SEDANG HITS & VIRAL!!!  api  api 
 api  api  EVENT SLOT SCATTER DAN PERKALIAN TERBESAR  api  api 
PROMO HANYA BULAN INI SAJA !!! COBA DEPO 50K PASTIKAN MAXWIN api  api 
IDNSCORE 𝗠𝗘𝗠𝗕𝗘𝗥𝗜𝗞𝗔𝗡 𝗕𝗢𝗡𝗨𝗦 𝟭𝟬𝟬% 𝗦𝗟𝗢𝗧  DAFTAR :
https://magic.ly/LGOGOAL_PROFIT_VIP
 api  api INFO BARU BOSS Q,NI LINK YG @SEDANG HITS & VIRAL!!!  api  api 
 api  api  EVENT SLOT SCATTER DAN PERKALIAN TERBESAR  api  api 
PROMO HANYA BULAN INI SAJA !!! COBA DEPO 50K PASTIKAN MAXWIN api  api 
IDNSCORE 𝗠𝗘𝗠𝗕𝗘𝗥𝗜𝗞𝗔𝗡 𝗕𝗢𝗡𝗨𝗦 𝟭𝟬𝟬% 𝗦𝗟𝗢𝗧  DAFTAR :
{LINK}
API API Info Baru Bos Q, Ni Link yang @sedang Hits & Viral !!!  api 
 API API Slot Slot Scatter dan perkalian API terbesar 
Promo hanya bulan ini !!! Coba Depo 50k Pastikan Api Maxwin Api 
IDNScore 𝗠𝗘𝗠𝗕𝗘𝗥𝗜𝗞𝗔𝗡 𝗕𝗢𝗡𝗨𝗦 𝟭𝟬𝟬% 𝗦𝗟𝗢𝗧 Daftar:
{LINK}
 api  api info baru boss q,ni link yg @sedang hits & viral!!!  api  api 
 api  api  event slot scatter dan perkalian terbesar  api  api 
promo hanya bulan ini saja !!! coba depo 50k pastikan maxwin api  api 
i

In [ ]:
# 4. Load data and run each preprocessing step with separate JSON output
import json

async def run_preprocessing_steps_separate_files():
    stop_words = set(stopwords.words('indonesian'))
    with open('fetched_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    # Step names and empty lists for results
    step_names = [
        'emoji', 'link', 'back_translation', 'lowercase', 'stopwords', 'slang',
        'extra_chars', 'usernames', 'numbers', 'punctuation', 'unidecode'
    ]
    step_results = {name: [] for name in step_names}
    for item in data:
        text = item['text']
        # 1. Back translation
        text1 = await back_translate(text)
        step_results['back_translation'].append(text1)
        # 2. Convert emojis to phrases
        text2 = convert_emojis(text1)
        step_results['emoji'].append(text2)
        # 3. Replace links with {LINK}
        text3 = replace_links(text2)
        step_results['link'].append(text3)
        # 4. Lowercase
        text4 = lowercase(text3)
        step_results['lowercase'].append(text4)
        # 5. Remove stopwords
        tokens = text4.split()  # Using simple split instead of word_tokenize for consistency
        text5 = ' '.join([w for w in tokens if w.lower() not in stop_words])
        step_results['stopwords'].append(text5)
        # 6. Replace slang words
        text6 = replace_slang(text5, slang_dict)
        step_results['slang'].append(text6)
        # 7. Remove extra characters
        text7 = remove_extra_chars(text6)
        step_results['extra_chars'].append(text7)
        # 8. Remove usernames
        text8 = remove_usernames(text7)
        step_results['usernames'].append(text8)
        # 9. Remove numbers
        text9 = remove_numbers(text8)
        step_results['numbers'].append(text9)
        # 10. Remove punctuation
        text10 = remove_punctuation(text9)
        step_results['punctuation'].append(text10)
        # 11. Unidecode (normalize unicode to ASCII)
        text11 = normalize_text(text10)
        step_results['unidecode'].append(text11)
    # Save each step's results to a separate JSON file
    for name in step_names:
        filename = f'fetched_data_{name}.json'
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(step_results[name], f, ensure_ascii=False, indent=2)
        print(f'Saved {name} results to {filename}')

await run_preprocessing_steps_separate_files()


CancelledError: 